# Llama 3.1 Model Comparison for Code Repair

This notebook compares the performance of three Llama 3.1 models (1B, 3B, 8B) on code repair tasks.

**Models Compared:**
- **Llama 3.1 1B**: Smallest, fastest, lowest memory (~2GB GPU)
- **Llama 3.1 3B**: Balanced, moderate quality (~8GB GPU)
- **Llama 3.1 8B**: Largest, best quality (~16GB GPU)

**Metrics:**
- Pass Rate: % of instances where model generates correct patches
- Format Correctness: % of valid SEARCH/REPLACE format
- Inference Speed: Average time per instance
- Throughput: Tokens per second

## Setup: Configure Cluster Endpoint

First, specify your cluster inference server endpoint. This could be:
- vLLM: `http://cluster.example.com:8000/v1`
- Together AI: `https://api.together.xyz/v1`
- Fireworks: `https://api.fireworks.ai/inference/v1`
- Local Ollama: `http://localhost:11434/v1`

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project to path
sys.path.insert(0, '/path/to/swerl')  # Update this path

from evaluation.compare_models import ModelComparison
from utils.io_utils import read_yaml

# Configure your cluster endpoint
CLUSTER_ENDPOINT = "http://localhost:8000/v1"  # TODO: Update this
CONFIG_PATH = "configs/llama_models.yaml"
TEST_FILE = "data/processed/test.jsonl"
OUTPUT_DIR = "evaluation/comparison_results"

print(f"Cluster Endpoint: {CLUSTER_ENDPOINT}")
print(f"Config: {CONFIG_PATH}")
print(f"Test File: {TEST_FILE}")

## Load Configuration

In [ ]:
# Load model configurations
config = read_yaml(CONFIG_PATH)

print("\nModel Configurations:")
print("="*70)

for model_key, model_config in config["models"].items():
    print(f"\n{model_key.upper()}")
    print(f"  Name: {model_config['name']}")
    print(f"  Parameters: {model_config['params_billions']}B")
    print(f"  Min GPU Memory: {model_config['hardware']['min_gpu_memory_gb']}GB")
    print(f"  Batch Size: {model_config['hardware']['recommended_batch_size']}")
    print(f"  Use Case: {model_config['use_case']}")

## Quick Test (Optional)

Test a single instance with all three models to verify cluster is working.

In [ ]:
from utils.llama_client import get_llama_client
from utils.io_utils import read_jsonl

# Load first test instance
test_instances = read_jsonl(TEST_FILE)
test_instance = test_instances[0]

print(f"Test Instance: {test_instance['instance_id']}")
print(f"Problem: {test_instance['problem_statement'][:100]}...")

# Quick test with 1B model
print("\nTesting with Llama 3.1 1B...")
client_1b = get_llama_client(
    endpoint_url=CLUSTER_ENDPOINT,
    model_name="meta-llama/Llama-3.1-1B-Instruct",
)

try:
    # Simple test message
    test_messages = [
        {"role": "user", "content": "What is 2+2?"}
    ]
    
    result = client_1b.call(test_messages, max_tokens=50)
    print(f"✓ Cluster is responding!")
    print(f"Response: {result}")
except Exception as e:
    print(f"✗ Cluster error: {e}")
    print(f"Make sure your cluster is running at {CLUSTER_ENDPOINT}")

## Run Full Comparison

This will evaluate all three models on your test set. Depending on test size, this may take 10-60 minutes.

In [ ]:
# Initialize comparison
comparison = ModelComparison(
    config_path=CONFIG_PATH,
    test_file=TEST_FILE,
    output_dir=OUTPUT_DIR,
)

print(f"Loaded {comparison.results['num_instances']} test instances")
print("\nStarting model evaluation...")
print("This may take a while depending on test set size.\n")

# Run comparison on all three models
# Optional: Use num_instances to test on a subset first (e.g., 20 instances)
results = comparison.run_comparison(
    cluster_endpoint=CLUSTER_ENDPOINT,
    models=["llama_1b", "llama_3b", "llama_8b"],
    num_instances=None,  # Set to 20 for quick test
)

print("\n✓ Evaluation complete!")

## Save Results

In [ ]:
# Save results to JSON
output_file = comparison.save_results()
print(f"Results saved to: {output_file}")

# Print summary
comparison.print_summary()

## Results Analysis

Parse and visualize the comparison results.

In [ ]:
# Extract metrics for comparison
metrics_data = []

for model_key, model_results in results["models"].items():
    summary = model_results["summary"]
    metrics_data.append({
        "Model": model_key,
        "Pass Rate (%)": summary["pass_rate"],
        "Format Correct (%)": summary["format_correctness"],
        "Avg Time (ms)": summary["average_time_per_instance_ms"],
        "Tokens/sec": summary["tokens_per_second"],
        "Total Errors": summary["errors"],
    })

df_metrics = pd.DataFrame(metrics_data)
print("\nPerformance Metrics:")
print(df_metrics.to_string(index=False))

## Visualization: Pass Rate Comparison

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Pass Rate Comparison
ax = axes[0, 0]
models = df_metrics["Model"]
pass_rates = df_metrics["Pass Rate (%)"]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax.bar(models, pass_rates, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Pass Rate (%)', fontsize=11, fontweight='bold')
ax.set_title('Code Repair Accuracy', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, pass_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.1f}%', ha='center', va='bottom', fontweight='bold')

# 2. Format Correctness
ax = axes[0, 1]
format_correct = df_metrics["Format Correct (%)"]
bars = ax.bar(models, format_correct, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Format Correctness (%)', fontsize=11, fontweight='bold')
ax.set_title('SEARCH/REPLACE Format Validity', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, format_correct):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.1f}%', ha='center', va='bottom', fontweight='bold')

# 3. Inference Speed (lower is better)
ax = axes[1, 0]
avg_times = df_metrics["Avg Time (ms)"]
bars = ax.bar(models, avg_times, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Time (milliseconds)', fontsize=11, fontweight='bold')
ax.set_title('Inference Speed (lower is better)', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, avg_times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.0f}ms', ha='center', va='bottom', fontweight='bold')

# 4. Throughput (tokens per second)
ax = axes[1, 1]
throughputs = df_metrics["Tokens/sec"]
bars = ax.bar(models, throughputs, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Tokens per Second', fontsize=11, fontweight='bold')
ax.set_title('Inference Throughput', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, throughputs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparison_metrics.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Chart saved to {OUTPUT_DIR}/comparison_metrics.png")

## Quality vs Speed Trade-off

In [ ]:
# Create scatter plot: Quality (Pass Rate) vs Speed (Throughput)
fig, ax = plt.subplots(figsize=(10, 6))

# Model sizes
sizes = [1, 3, 8]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
model_names = ['Llama 1B', 'Llama 3B', 'Llama 8B']

# Plot
for i, (model, size, color) in enumerate(zip(models, sizes, colors)):
    pass_rate = df_metrics.iloc[i]['Pass Rate (%)']
    throughput = df_metrics.iloc[i]['Tokens/sec']
    
    ax.scatter(throughput, pass_rate, s=size*300, alpha=0.6, c=color, 
              edgecolors='black', linewidth=2, label=model_names[i])
    ax.annotate(model_names[i], (throughput, pass_rate), 
               xytext=(5, 5), textcoords='offset points', fontweight='bold')

ax.set_xlabel('Throughput (Tokens/sec)', fontsize=12, fontweight='bold')
ax.set_ylabel('Code Repair Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Quality vs Speed Trade-off', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/quality_vs_speed.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Chart saved to {OUTPUT_DIR}/quality_vs_speed.png")

## Detailed Error Analysis

In [ ]:
# Analyze failures by type
print("\nDetailed Error Analysis:")
print("="*70)

for model_key, model_results in results["models"].items():
    print(f"\n{model_key.upper()}")
    
    results_list = model_results["results"]
    
    # Count failure types
    total = len(results_list)
    errors = sum(1 for r in results_list if "error" in r)
    format_fail = sum(1 for r in results_list if not r.get("format_correct", False))
    correctness_fail = sum(1 for r in results_list if not r.get("correctness", False))
    
    print(f"  Total: {total}")
    print(f"  Errors: {errors} ({errors/total*100:.1f}%)")
    print(f"  Format Issues: {format_fail} ({format_fail/total*100:.1f}%)")
    print(f"  Correctness Issues: {correctness_fail} ({correctness_fail/total*100:.1f}%)")
    
    # Sample errors
    error_samples = [r for r in results_list if "error" in r][:2]
    if error_samples:
        print(f"  Sample Errors:")
        for sample in error_samples:
            error_msg = str(sample.get("error", ""))[:80]
            print(f"    - {error_msg}")

## Recommendation

Based on the comparison results, here's our recommendation:

In [ ]:
# Determine best model for each use case
print("\nRECOMMENDATIONS BY USE CASE")
print("="*70)

# Get metrics
metrics_dict = {}
for _, row in df_metrics.iterrows():
    metrics_dict[row['Model']] = row

# Best accuracy
best_accuracy = max(metrics_dict.items(), key=lambda x: x[1]['Pass Rate (%)'])
print(f"\n✓ BEST ACCURACY: {best_accuracy[0]}")
print(f"  Pass Rate: {best_accuracy[1]['Pass Rate (%)']:.1f}%")
print(f"  → Use for: Mission-critical patches, production code")

# Best speed
best_speed = min(metrics_dict.items(), key=lambda x: x[1]['Avg Time (ms)'])
print(f"\n⚡ FASTEST: {best_speed[0]}")
print(f"  Avg Time: {best_speed[1]['Avg Time (ms)']:.1f}ms")
print(f"  → Use for: Real-time scenarios, batch processing")

# Best balance
# Normalize metrics to 0-1 and compute balance score
max_accuracy = max(m['Pass Rate (%)'] for m in metrics_dict.values())
min_time = min(m['Avg Time (ms)'] for m in metrics_dict.values())

balance_scores = {}
for model, metrics in metrics_dict.items():
    accuracy_norm = metrics['Pass Rate (%)'] / max_accuracy
    speed_norm = min_time / metrics['Avg Time (ms)']
    balance_scores[model] = (accuracy_norm + speed_norm) / 2

best_balance = max(balance_scores.items(), key=lambda x: x[1])
print(f"\n⚖️  BEST BALANCE: {best_balance[0]}")
print(f"  Score: {best_balance[1]:.3f}")
print(f"  → Use for: General purpose, production with resource constraints")

print(f"\n" + "="*70)

## Export Results for Report

Create a summary table for your final report.

In [ ]:
# Create formatted summary table
summary_df = df_metrics[['Model', 'Pass Rate (%)', 'Format Correct (%)', 
                         'Avg Time (ms)', 'Tokens/sec']].copy()
summary_df.columns = ['Model', 'Accuracy (%)', 'Valid Format (%)', 
                      'Latency (ms)', 'Throughput (tok/s)']

# Round values
for col in ['Accuracy (%)', 'Valid Format (%)', 'Latency (ms)', 'Throughput (tok/s)']:
    summary_df[col] = summary_df[col].round(2)

print("\nSUMMARY TABLE FOR REPORT:")
print(summary_df.to_string(index=False))

# Save as CSV
summary_df.to_csv(f"{OUTPUT_DIR}/summary_table.csv", index=False)
print(f"\n✓ Summary table saved to {OUTPUT_DIR}/summary_table.csv")